# E7a: repeated cross-validation of the whole pipeline

**Revision plan, 00 log section 20 (approved 17 Oct 2026).** Answers **Reviewer 4 point 6** ("test set small (n = 56),
wide CIs; use repeated CV or report by tumour subgroup") — the subgroup part is in E47.

- **5 repeats × 5-fold** stratified cross-validation of the 222 development patients. **Repeat 0 = the original folds**
  (the saved E3 phase-dropout CNN and E2 radiomics are reused and checked); **repeats 1–4 = new random folds** (seeds 1–4).
- In every repeat the **whole pipeline is retrained**: phase-dropout CNN (E3 protocol, 40 epochs, inner validation split),
  E2 radiomics without shape features, and the CNN + radiomics and full-fusion meta-learners on that repeat's out-of-fold
  predictions.
- Reported: out-of-fold macro-AUC per repeat (pooled and fold mean), the spread over all 25 folds, the held-out (56)
  macro-AUC of each repeat's 5-model ensemble, and the paired differences between models per repeat.

Nothing is tuned; the 56 held-out patients are never used for training or selection.

**Runtime:** **A100 GPU**, about 1.5–2 hours (20 CNN trainings). **Resume:** set `RESUME_RUN_ID` in Cell 1 to the stopped
run's time and run all again; trained folds are reused. **Safety:** outputs go to a new folder
`liver_project/revision_2026-10/E7a_<time>/`.

In [ ]:
# Cell 1 — Setup: installs, paths, input checks, output folder
import os, sys, json, time, random, hashlib, platform, datetime, warnings, subprocess
from pathlib import Path
warnings.filterwarnings('ignore')
RESUME_RUN_ID = None   # to continue a stopped run, put its time here, e.g. '20261018T020304Z'

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai', 'lightgbm==4.6.0'], check=True)
else:
    ROOT = Path(os.environ['E7A_TEST_ROOT'])

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import lightgbm as lgb
from monai.networks.nets import resnet18
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import f_classif
from sklearn.metrics import roc_auc_score

PROJECT_DIR = ROOT / 'liver_project'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
SEED = 42
P3 = ['p_HCC', 'p_ICC', 'p_CHCC']
NUM_EPOCHS = int(os.environ.get('E7A_TEST_EPOCHS', 40))
BATCH_SIZE, LR, WD = 4, 1e-4, 1e-5
N_REPEATS = 5
INPUTS = {
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_test_meta': (E2_DIR / 'E2_fusion_test_meta_features_noshape.csv', 'fbb946f641a050f66b093564141a8e103e1c4e5c651d6abeb51042293ab8cb2f'),
    'e2_radiomics_oof': (E2_DIR / 'E2_radiomics_noshape_oof_predictions.csv', '21057b42933293a7d45cbefbb6f58dfe89458064a42fff689abcbc826fe28cb6'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()
problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}'); continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
tar_path = PROJECT_DIR / 'processed.tar'
if not tar_path.exists() or (IN_COLAB and tar_path.stat().st_size != 402616320):
    problems.append(f'missing or changed size: {tar_path}')
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
WORKERS = 4 if IN_COLAB else 0
import sklearn, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__, 'lightgbm': lgb.__version__,
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__,
            'device': torch.cuda.get_device_name(0) if USE_AMP else 'cpu'}
RUN_ID = RESUME_RUN_ID or datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'E7a_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=bool(RESUME_RUN_ID))
print('Versions:', VERSIONS, '| output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Crops, labels and the 5 fold assignments (repeat 0 = original folds; repeats 1-4 = new stratified folds)
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(tar_path), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E7A_TEST_PROCESSED'])
split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
CT, LABEL = {}, {}
for pid in dev_ids + test_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        CT[pid], LABEL[pid] = z['ct'].astype(np.uint8), int(z['label'])
y_dev = np.array([LABEL[p] for p in dev_ids])
y_test = np.array([LABEL[p] for p in test_ids])
REPEATS = {0: [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]}
for r in range(1, N_REPEATS):
    folds = []
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=r).split(dev_ids, y_dev):
        folds.append({'train_ids': [dev_ids[i] for i in tr], 'val_ids': [dev_ids[i] for i in va]})
    REPEATS[r] = folds
(OUT_DIR / 'E7a_fold_assignments.json').write_text(json.dumps({str(r): f for r, f in REPEATS.items()}))
for r, folds in REPEATS.items():
    assert sorted(sum((f['val_ids'] for f in folds), [])) == sorted(dev_ids), r
print('Repeats:', {r: [len(f['val_ids']) for f in folds] for r, folds in REPEATS.items()})

In [ ]:
# Cell 3 — E3 phase-dropout training code (unchanged protocol)
def zscore4(x4):
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def augment(vol4):
    if random.random() < 0.5:
        vol4 = vol4[:, ::-1].copy()
    if random.random() < 0.5:
        vol4 = vol4[:, :, ::-1].copy()
    if random.random() < 0.5:
        for c in range(vol4.shape[0]):
            scale = 1.0 + (random.random() - 0.5) * 0.2
            shift = (random.random() - 0.5) * 0.2
            mask = vol4[c] != 0
            vol4[c][mask] = vol4[c][mask] * scale + shift
    return vol4

def sample_kept_phases():
    if random.random() < 0.5:
        return (0, 1, 2, 3)
    while True:
        kept = tuple(c for c in range(4) if random.random() >= 0.5)
        if kept:
            return kept

def keep_only(vol4, kept):
    out = vol4.copy()
    for c in range(4):
        if c not in kept:
            out[c] = 0.0
    return out

class CropDataset(Dataset):
    def __init__(self, ids, train=False):
        self.ids, self.train = list(ids), train
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, index):
        pid = self.ids[index]
        vol4 = zscore4(CT[pid].astype(np.float32))
        if self.train:
            vol4 = augment(vol4)
            kept = sample_kept_phases()
            if kept != (0, 1, 2, 3):
                vol4 = keep_only(vol4, kept)
        return torch.from_numpy(np.ascontiguousarray(vol4, dtype=np.float32)), LABEL[pid], pid

def seed_worker(worker_id):
    np.random.seed(torch.initial_seed() % 2 ** 32)

def make_model():
    return resnet18(spatial_dims=3, n_input_channels=4, num_classes=3, feed_forward=True)

def class_weights_for(ids):
    counts = np.bincount([LABEL[p] for p in ids], minlength=3).astype(np.float32)
    return torch.from_numpy(counts.sum() / (3 * counts.clip(min=1))).float()

def macro_auc(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))

@torch.no_grad()
def predict(model, ids, batch_size=8):
    model.eval()
    loader = DataLoader(CropDataset(ids), batch_size=batch_size, shuffle=False, num_workers=2 if IN_COLAB else 0, pin_memory=USE_AMP)
    probs, pids = [], []
    for x, _, batch_ids in loader:
        probs.append(torch.softmax(model(x.to(DEVICE, non_blocking=True)).float().cpu(), dim=1).numpy())
        pids.extend(batch_ids)
    return pd.DataFrame(np.concatenate(probs), index=pids, columns=P3)

def train_fold(r, f):
    path = OUT_DIR / f'repeat_{r}' / f'fold_{f}.pt'
    if path.exists():
        print(f'  repeat {r} fold {f}: already trained, loading.')
        return
    path.parent.mkdir(exist_ok=True)
    outer_train = list(REPEATS[r][f]['train_ids'])
    inner_train, inner_val = train_test_split(outer_train, test_size=0.2, random_state=SEED + 10 * r + f,
                                              stratify=[LABEL[p] for p in outer_train])
    seed = SEED + 10 * r + f
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if USE_AMP:
        torch.cuda.manual_seed_all(seed)
    loader = DataLoader(CropDataset(inner_train, train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS,
                        pin_memory=USE_AMP, drop_last=True, persistent_workers=WORKERS > 0, worker_init_fn=seed_worker)
    model = make_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights_for(inner_train).to(DEVICE))
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    y_inner = np.array([LABEL[p] for p in inner_val])
    best_auc, best_state, best_epoch = -1.0, None, -1
    t0 = time.time()
    for epoch in range(NUM_EPOCHS):
        model.train()
        for x, y, _ in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        auc = macro_auc(y_inner, predict(model, inner_val).loc[inner_val].to_numpy())
        if auc > best_auc:
            best_auc, best_epoch = auc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    torch.save({'state_dict': best_state, 'repeat': r, 'fold': f, 'best_epoch': best_epoch, 'best_inner_val_auc': best_auc}, path)
    print(f'  repeat {r} fold {f}: best inner-val AUC {best_auc:.4f} at epoch {best_epoch} ({(time.time() - t0) / 60:.1f} min)', flush=True)
    del model
    if USE_AMP:
        torch.cuda.empty_cache()

def load_state(path):
    model = make_model().to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=False)['state_dict'])
    model.eval()
    return model

In [ ]:
# Cell 4 — Train the phase-dropout CNN for repeats 1-4 (repeat 0 = the saved E3 models)
t0 = time.time()
for r in range(1, N_REPEATS):
    for f in range(5):
        train_fold(r, f)
print(f'Training done in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Cell 5 — CNN out-of-fold and held-out predictions per repeat; radiomics (E2 recipe) per repeat
d_saved = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
CNN = {0: (d_saved.query("pattern == 'complete' and split == 'oof'").set_index('patient_id').loc[dev_ids, P3],
           d_saved.query("pattern == 'complete' and split == 'test'").set_index('patient_id').loc[test_ids, P3])}
for r in range(1, N_REPEATS):
    models = [load_state(OUT_DIR / f'repeat_{r}' / f'fold_{f}.pt') for f in range(5)]
    oof = pd.concat([predict(models[f], REPEATS[r][f]['val_ids']) for f in range(5)]).loc[dev_ids]
    test = sum(predict(m, test_ids).loc[test_ids] for m in models) / 5
    CNN[r] = (oof, test)
    del models

K_TOP_FEATURES, CORR_THRESHOLD = 30, 0.95
def select_features_train(X_tr, y_tr, feature_names, k=K_TOP_FEATURES, corr_thresh=CORR_THRESHOLD):   # Week 5 / E2
    stds = np.nanstd(X_tr, axis=0)
    keep = stds > 1e-10
    X_kept = X_tr[:, keep]
    kept_idx_global = np.where(keep)[0]
    X_nan_to_num = np.nan_to_num(X_kept, nan=np.nanmedian(X_kept))
    corr = np.abs(np.corrcoef(X_nan_to_num.T))
    remove = np.zeros(corr.shape[0], dtype=bool)
    for i in range(corr.shape[0]):
        if remove[i]:
            continue
        for j in range(i + 1, corr.shape[0]):
            if remove[j]:
                continue
            if corr[i, j] > corr_thresh:
                remove[j] = True
    kept_idx_after_corr = kept_idx_global[~remove]
    X_for_f = SimpleImputer(strategy='median').fit_transform(X_tr[:, kept_idx_after_corr])
    F, _ = f_classif(X_for_f, y_tr)
    order = np.argsort(F)[::-1]
    final_idx = kept_idx_after_corr[order[:min(k, len(order))]]
    return final_idx, [feature_names[i] for i in final_idx]

def radiomics_repeat(folds, df, cols):
    oof, test = [], []
    test_X = df.set_index('patient_id').loc[test_ids, cols].to_numpy(dtype=np.float32)
    for s in folds:
        tr = df[df['patient_id'].isin(s['train_ids'])]
        va = df.set_index('patient_id').loc[s['val_ids']]
        X_tr, y_tr = tr[cols].to_numpy(dtype=np.float32), tr['label'].to_numpy()
        idx, _ = select_features_train(X_tr, y_tr, cols)
        imp = SimpleImputer(strategy='median').fit(X_tr[:, idx])
        sc = StandardScaler().fit(imp.transform(X_tr[:, idx]))
        clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, num_leaves=15, min_child_samples=5,
                                 reg_alpha=0.1, reg_lambda=0.1, class_weight='balanced', random_state=42, verbose=-1)
        clf.fit(sc.transform(imp.transform(X_tr[:, idx])), y_tr)
        oof.append(pd.DataFrame(clf.predict_proba(sc.transform(imp.transform(va[cols].to_numpy(dtype=np.float32)[:, idx]))), index=s['val_ids'], columns=P3))
        test.append(clf.predict_proba(sc.transform(imp.transform(test_X[:, idx]))))
    return pd.concat(oof).loc[dev_ids], pd.DataFrame(np.mean(test, axis=0), index=test_ids, columns=P3)

internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
POOL = [c for c in internal.columns if c not in ('patient_id', 'label') and '_original_shape_' not in c]
RADIO = {r: radiomics_repeat(REPEATS[r], internal, POOL) for r in range(N_REPEATS)}
e2_oof = pd.read_csv(INPUTS['e2_radiomics_oof'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids, P3]
GATE_R0 = {'radiomics_oof_vs_E2': float(np.abs(RADIO[0][0].to_numpy() - e2_oof.to_numpy()).max())}
print('Repeat 0 radiomics reproduces E2 (max difference):', GATE_R0['radiomics_oof_vs_E2'])
if IN_COLAB and GATE_R0['radiomics_oof_vs_E2'] > 1e-6:
    raise RuntimeError('STOP — repeat 0 radiomics does not reproduce E2. Send this output.')

In [ ]:
# Cell 6 — Fusions per repeat (meta-learners on that repeat's out-of-fold predictions; E2 recipe)
W4, W5 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC'], ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
meta = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids]
tmeta = pd.read_csv(INPUTS['e2_test_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids]
def prepare(X):
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X
def meta_cv(columns, oof, test, folds):
    fold_of = {p: f for f, s in enumerate(folds) for p in s['val_ids']}
    oof_p, test_p = pd.DataFrame(index=dev_ids, columns=P3, dtype=float), []
    for f, s in enumerate(folds):
        tr = [p for p in dev_ids if fold_of[p] != f]
        X = prepare(oof.loc[tr, columns].to_numpy())
        imp = SimpleImputer(strategy='median').fit(X)
        sc = StandardScaler().fit(imp.transform(X))
        m = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED).fit(sc.transform(imp.transform(X)), oof.loc[tr, 'label'].astype(int))
        oof_p.loc[s['val_ids']] = m.predict_proba(sc.transform(imp.transform(prepare(oof.loc[s['val_ids'], columns].to_numpy()))))
        test_p.append(m.predict_proba(sc.transform(imp.transform(prepare(test[columns].to_numpy())))))
    return oof_p, pd.DataFrame(np.mean(test_p, axis=0), index=test_ids, columns=P3)
PRED = {}   # (repeat, model) -> (oof, test)
for r in range(N_REPEATS):
    oof, test = meta.copy(), tmeta.copy()
    oof[W4], test[W4] = CNN[r][0].loc[dev_ids].to_numpy(), CNN[r][1].loc[test_ids].to_numpy()
    oof[W5], test[W5] = RADIO[r][0].loc[dev_ids].to_numpy(), RADIO[r][1].loc[test_ids].to_numpy()
    PRED[(r, 'phase_dropout_CNN')] = CNN[r]
    PRED[(r, 'radiomics_noshape')] = RADIO[r]
    PRED[(r, 'cnn_plus_radiomics')] = meta_cv(W4 + W5, oof, test, REPEATS[r])
    PRED[(r, 'full_fusion')] = meta_cv(W4 + W5 + ['age', 'gender'], oof, test, REPEATS[r])
GATE_R0['repeat0_test_auc'] = {m: round(macro_auc(y_test, PRED[(0, m)][1].to_numpy()), 3) for m in ['phase_dropout_CNN', 'radiomics_noshape', 'cnn_plus_radiomics', 'full_fusion']}
print('Repeat 0 held-out macro-AUC (must equal E3/E47: 0.919, 0.902, 0.952, 0.951):', GATE_R0['repeat0_test_auc'])
if IN_COLAB and any(abs(GATE_R0['repeat0_test_auc'][m] - v) > 0.002 for m, v in
                    {'phase_dropout_CNN': 0.919, 'radiomics_noshape': 0.902, 'cnn_plus_radiomics': 0.952, 'full_fusion': 0.951}.items()):
    raise RuntimeError('STOP — repeat 0 does not reproduce the saved results. Send this output.')

In [ ]:
# Cell 7 — Results: per repeat, over all 25 folds, held-out per repeat, paired differences; saved outputs
MODELS = ['phase_dropout_CNN', 'radiomics_noshape', 'cnn_plus_radiomics', 'full_fusion']
rows, fold_rows = [], []
for r in range(N_REPEATS):
    for m in MODELS:
        oof, test = PRED[(r, m)]
        folds = [macro_auc([LABEL[p] for p in s['val_ids']], oof.loc[s['val_ids']].to_numpy()) for s in REPEATS[r]]
        fold_rows += [{'repeat': r, 'fold': f, 'model': m, 'auc': a} for f, a in enumerate(folds)]
        rows.append({'repeat': r, 'model': m, 'oof_pooled_auc': macro_auc(y_dev, oof.loc[dev_ids].to_numpy()), 'fold_mean_auc': float(np.mean(folds)),
                     'fold_sd_auc': float(np.std(folds)), 'held_out_auc': macro_auc(y_test, test.loc[test_ids].to_numpy())})
PER_REPEAT, PER_FOLD = pd.DataFrame(rows), pd.DataFrame(fold_rows)
SUMMARY = []
for m in MODELS:
    pr, pf = PER_REPEAT[PER_REPEAT['model'] == m], PER_FOLD[PER_FOLD['model'] == m]['auc']
    SUMMARY.append({'model': m, 'oof_pooled_mean': pr['oof_pooled_auc'].mean(), 'oof_pooled_sd': pr['oof_pooled_auc'].std(ddof=1),
                    'all25_folds_mean': pf.mean(), 'all25_folds_sd': pf.std(ddof=1), 'all25_p2_5': pf.quantile(.025), 'all25_p97_5': pf.quantile(.975),
                    'held_out_mean': pr['held_out_auc'].mean(), 'held_out_min': pr['held_out_auc'].min(), 'held_out_max': pr['held_out_auc'].max()})
SUMMARY = pd.DataFrame(SUMMARY)
DIFFS = []
for a, b in [('full_fusion', 'cnn_plus_radiomics'), ('cnn_plus_radiomics', 'radiomics_noshape'), ('cnn_plus_radiomics', 'phase_dropout_CNN'),
             ('phase_dropout_CNN', 'radiomics_noshape')]:
    pa, pb = PER_REPEAT.set_index(['repeat', 'model'])['oof_pooled_auc'], PER_REPEAT.set_index(['repeat', 'model'])['held_out_auc']
    d_oof = [pa[(r, a)] - pa[(r, b)] for r in range(N_REPEATS)]
    d_test = [pb[(r, a)] - pb[(r, b)] for r in range(N_REPEATS)]
    DIFFS.append({'model_a': a, 'model_b': b, 'oof_diff_mean': float(np.mean(d_oof)), 'oof_diff_min': float(np.min(d_oof)), 'oof_diff_max': float(np.max(d_oof)),
                  'repeats_a_better_oof': int(sum(d > 0 for d in d_oof)), 'held_out_diff_mean': float(np.mean(d_test)),
                  'repeats_a_better_held_out': int(sum(d > 0 for d in d_test))})
DIFFS = pd.DataFrame(DIFFS)
for name, frame in [('E7a_per_repeat', PER_REPEAT), ('E7a_per_fold', PER_FOLD), ('E7a_summary', SUMMARY), ('E7a_paired_differences', DIFFS)]:
    frame.to_csv(OUT_DIR / f'{name}.csv', index=False)
long = []
for (r, m), (oof, test) in PRED.items():
    for split_name, frame in [('oof', oof), ('test', test)]:
        t = frame.copy(); t.insert(0, 'patient_id', t.index); t.insert(0, 'split', split_name); t.insert(0, 'model', m); t.insert(0, 'repeat', r); long.append(t)
pd.concat(long, ignore_index=True).to_csv(OUT_DIR / 'E7a_predictions.csv', index=False)
summary = {'run_id': RUN_ID, 'notebook': 'E7a_repeated_cross_validation.ipynb', 'versions': VERSIONS, 'input_sha256': input_hashes,
           'gates': GATE_R0, 'epochs': NUM_EPOCHS, 'repeats': N_REPEATS, 'protocol': '00 log section 20 (approved 17 Oct 2026)'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest_out = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)} for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest_out).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E7a_{RUN_ID} | epochs {NUM_EPOCHS} | gates: {GATE_R0}')
print('\n--- per repeat ---')
print(PER_REPEAT.round(3).to_string(index=False))
print('\n--- summary over repeats and all 25 folds ---')
print(SUMMARY.round(3).to_string(index=False))
print('\n--- paired differences per repeat ---')
print(DIFFS.round(3).to_string(index=False))
print('==============================================================')

In [ ]:
# Final cell — make sure every result file has reached Google Drive, then shut the runtime down (stops compute units)
import time
from google.colab import drive, runtime
drive.flush_and_unmount()
print('All files are saved in Drive. The runtime shuts down in 30 seconds; the outputs above stay visible on this page.')
time.sleep(30)
runtime.unassign()